In [ ]:
import dai
import pandas as pd
import numpy as np

# ============================================================
# factor_quality_shape_gate_v1: 高A factorlib 基础 × 日内分布矩门 ⭐
# ============================================================
# 基于复盘结论组合两个已验证高A通道:
#   longhorizon_v1_fixed (A=0.748, B=0.595): factorlib 综合高A
#   intraday_shape (A=0.610, B=0.677): 日内收益分布矩 (峰度/偏度)
# 设计: 用日内分布矩做"正交门", 避免 highA_gated 用签名订单流门的失败
#   (签名订单流与全场高相关 → B=0.179; 日内分布矩与全场低相关)
#   raw = base × (0.25 + positive(base × shape))
#   base  = factorlib 综合 (A 引擎)
#   shape = 日内分布矩确认 (正交门, 只在一致时放大)
# 经济学: 高A基础信号 + 日内信息脉冲确认 → 双向提升
#
# 安全设计 (对齐已验证通过模板):
#   factorlib 硬编码表名; bar1m 用 datasources["bar1m"]
#   全部日内计算, 聚合到日后再 shift(1) → PIT安全
#   全部跨日 shift(1) 在股票池 join 之前完成
#   涨跌停排除用前日收益
# ============================================================

def main(datasources, start_date, end_date):
    BUFFER_DAYS = 30
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=BUFFER_DAYS)).strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    # ---- 1. factorlib (A 引擎) ----
    sql_flib = f"""
        SELECT date, instrument,
            momentum_5, reversal_5, volatility_5,
            atr_14, bias_20, turn, list_days,
            roe_avg_ttm, roa_avg_ttm, gross_profit_rate_ttm, net_profit_rate_ttm,
            pe_ttm, pb, ps_ttm, debt_to_asset_lf,
            macd_hist_12_26_9, netflow_amount_rate_main, total_market_cap
        FROM bigalpha_2026_factorlib
    """
    flib = dai.query(sql_flib, filters={"date": [query_start, end_date]}, compression=True).df()
    if flib.empty:
        raise ValueError("Empty factorlib data")
    flib["date"] = pd.to_datetime(flib["date"])

    # ---- 2. bar1m (日内分布矩, 正交门) ----
    sql_bar = f"""
        SELECT date, instrument, close
        FROM {datasources["bar1m"]}
        WHERE close > 0
    """
    bar = dai.query(sql_bar, filters={"date": [query_start, end_date]}, compression=True).df()
    if bar.empty:
        raise ValueError("Empty bar1m data")
    bar["date"] = pd.to_datetime(bar["date"])
    bar["trading_day"] = bar["date"].dt.strftime("%Y-%m-%d")
    bar = bar.sort_values(["instrument", "date"])
    bar["min_ret"] = bar.groupby(["trading_day", "instrument"])["close"].pct_change()
    bar["min_ret"] = bar["min_ret"].replace([np.inf, -np.inf], np.nan)

    shape = bar.groupby(["trading_day", "instrument"]).agg(
        ret_std=("min_ret", "std"),
        ret_max=("min_ret", "max"),
        ret_min=("min_ret", "min"),
        ret_abs_mean=("min_ret", lambda x: x.abs().mean()),
        close_last=("close", "last"),
    ).reset_index()
    shape = shape.rename(columns={"trading_day": "date"})
    shape["date"] = pd.to_datetime(shape["date"])
    shape = shape.sort_values(["instrument", "date"])
    # 峰度代理 (max_abs / std), 偏度代理
    shape["peakiness"] = (shape["ret_max"].abs().clip(lower=0) + shape["ret_min"].abs().clip(lower=0)) / (shape["ret_std"] + 1e-8)
    shape["skew_proxy"] = (shape["ret_max"] + shape["ret_min"]) / (shape["ret_std"] + 1e-8)
    shape["vol_level"] = shape["ret_std"]
    # 前日
    g = shape.groupby("instrument")
    shape["prev_peak"] = g["peakiness"].shift(1)
    shape["prev_skew"] = g["skew_proxy"].shift(1)
    shape["prev_vol"] = g["vol_level"].shift(1)

    # ---- 3. 对齐股票池 ----
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])
    daily = pd.merge(stk_pool, flib, how="left", on=["date", "instrument"])
    daily = pd.merge(daily, shape[["date", "instrument", "prev_peak", "prev_skew", "prev_vol"]],
                     how="left", on=["date", "instrument"])
    daily = daily.sort_values(["instrument", "date"])

    # ---- 4. A 引擎: factorlib 综合 (同 longhorizon_v1) ----
    def cz(series):
        return series.groupby(daily["date"]).transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-8)
        )

    def cr(series):
        return series.groupby(daily["date"]).rank(pct=True)

    daily["log_turn"] = np.log1p(daily["turn"].fillna(0))
    daily["log_listdays"] = np.log1p(daily["list_days"].fillna(0))

    f_lowturn = -cz(daily["log_turn"])
    f_age = cz(daily["log_listdays"])
    f_mom_atr = cz(daily["momentum_5"].fillna(0) / (daily["atr_14"].fillna(0) + 1e-8))
    f_bias_rev = -cz(daily["bias_20"].fillna(0))
    f_lv = -cz(daily["volatility_5"].fillna(0))
    q_ranks = [cr(daily[c].fillna(0)) for c in ["roe_avg_ttm", "roa_avg_ttm", "gross_profit_rate_ttm", "net_profit_rate_ttm"]]
    f_quality = cz(sum(q_ranks) / len(q_ranks))
    v_ranks = [-cr(daily[c].fillna(0)) for c in ["pe_ttm", "pb", "ps_ttm", "debt_to_asset_lf"]]
    f_value = cz(sum(v_ranks) / len(v_ranks))
    f_macd = cz(daily["macd_hist_12_26_9"].fillna(0))
    f_flow = cz(daily["netflow_amount_rate_main"].fillna(0))
    f_qlv = cz((sum(q_ranks) / len(q_ranks)) * (-daily["volatility_5"].fillna(0)))

    daily["base"] = (
        0.16 * f_lowturn + 0.10 * f_age + 0.12 * f_mom_atr
        + 0.10 * f_bias_rev + 0.10 * f_lv + 0.12 * f_quality
        + 0.10 * f_value + 0.06 * f_macd + 0.06 * f_flow + 0.08 * f_qlv
    )
    daily["base"] = cz(daily["base"])

    # ---- 5. 正交门: 日内分布矩 ----
    daily["gate_peak"] = cz(daily["prev_peak"].fillna(0.5))
    daily["gate_skew"] = cz(daily["prev_skew"].fillna(0.5))

    # 条件乘积: 只在 base 与分布矩方向一致时放大
    daily["align_p"] = daily["base"] * daily["gate_peak"]
    daily["align_s"] = daily["base"] * daily["gate_skew"]
    daily["raw"] = daily["base"] * (0.5 + np.maximum(daily["align_p"], 0.0) * 0.5 + np.maximum(daily["align_s"], 0.0) * 0.5)

    # 截面 rank 输出
    daily["factor"] = daily.groupby("date", sort=False)["raw"].rank(pct=True)
    daily["factor"] = daily["factor"].fillna(0.5)

    # 输出
    result = daily[["date", "instrument", "factor"]].copy()
    result["date"] = pd.to_datetime(result["date"])
    result = result.dropna(subset=["factor"])
    result = result[
        (result["date"] >= pd.to_datetime(start_date))
        & (result["date"] <= pd.to_datetime(end_date))
    ]
    return result
